# Методы улучшения генерации
В прошлом уроке мы научились управлять генерацией на низком уровне — через параметры, влияющие на выбор токенов. В этом уроке поработаем на более высоком уровне: будем менять промпты, чтобы улучшить ответы модели. Чтобы сравнить все подходы, протестируем их на одном датасете и посчитаем метрики для каждого способа.

Для этого мы возьмём открытый датасет школьных задач T‑math (https://huggingface.co/datasets/t-tech/T-math) и чатовую модель `Qwen/Qwen3‑0.6B`. Шаг за шагом будем улучшать качество ответов.

## 1. Датасет
Любое обучение начинается с данных. Посмотрим на датасет. Нам интересны поля `question` — задача, `verifiable_answer` — числовой ответ без комментариев, `solutions` — возможные решения.

Наша цель — получать от модели правильные ответы на задачи. Но важно не только это: нам нужен единый формат вывода, чтобы автоматически проверять результаты и сравнивать подходы.

Например, модель может генерировать не только число, но и комментарии. В таком случае первое число в тексте может не быть окончательным ответом. Поэтому сразу договоримся о метриках и о формате ответа.

Допустим, мы хотим, чтобы модель всегда писала результаты так:
```
[SOLUTION] …краткое решение…
[ANSWER] …финальный ответ… 
```

Блок `[ANSWER]` будем проверять на точное совпадение с колонкой `verifiable_answer`. 

## 2. Подготовка метрик
В этом блоке мы опишем две метрики, которыми будем пользоваться весь урок: `accuracy` по извлечённому `[ANSWER]` и `format_rate` — долю примеров, где предсказание можно распарсить нашим шаблоном.

### Задание 1
Чтобы провести эксперименты быстрее, выполняйте задания этого урока на виртуальной машине с GPU.

Сначала создадим заготовку: функции `extract_answer`, `normalize` и `compute_metrics`. 
- В `extract_answer` мы ищем паттерн с маркерами `[SOLUTION]` и `[ANSWER]` и возвращаем текст ответа.
- В `normalize` извлекаем первое целое или дробное число.
- В `compute_metrics` считаем точность по всем примерам, где ответ удалось достать, и долю распарсенных результатов.

In [1]:
import re

def extract_answer(text: str):
    """
    Извлекает ответ из переданного текста.

    :param text: Текст, в котором будет производиться поиск. Может быть None.
    :return: Кортеж (bool, str | None), где bool — признак того, что ответ найден,
        а str | None — извлечённый ответ или None, если ничего не найдено.
    """
    if text is None:
        return False, None
    match = re.search(r"\[ANSWER\] (.+)", text)
    if match:
        return True, match.group(1)
    return False, None


def normalize(s: str) -> str:
    """
    Извлекает из строки первое целое или дробное число

    :param s: Исходная строка или None.
    :return: число типа float
    """
    if s is None:
        return None
    match = re.search(r"\d+\.?\d*", s)
    if match:
        return float(match.group())
    return None


def compute_metrics(preds, refs):
    parsed = 0
    correct = 0
    for p, r in zip(preds, refs):
        ok, val = extract_answer(p)
        if ok:
            parsed += 1
            if normalize(val) == normalize(r):
                correct += 1
    n = len(refs)
    return {
        "format_rate": parsed / n if n else 0.0,
        "accuracy": correct / n if n else 0.0,
        "count": n,
    }

# тесты
preds = ["[SOLUTION] 2+2=4\n[ANSWER] 4.0", "ответ без формата", "[ANSWER] число"]
refs = ["4", "что-то", "0"]
print(compute_metrics(preds, refs))
# ожидаемый результат {'format_rate': 0.6666666666666666, 
# 'accuracy': 0.3333333333333333,
# 'count': 3}

{'format_rate': 0.6666666666666666, 'accuracy': 0.3333333333333333, 'count': 3}


## 3. Бейзлайн: zero‑shot

Теперь перейдём к замерам модели. Чтобы зафиксировать текущую планку, начнём с самого простого: попросим модель дать короткий ответ без каких‑либо комментариев. Такой режим называется zero‑shot — мы не даём модели ни одного примера решения в промпте и полагаемся только на её предобученные знания. Так вы увидите качество «из коробки». 

Для этого будем вызывать уже знакомый вам метод `.generate()`, перед этим формируя задачу в формате чата.


### Задание 2
Выполните задание на ВМ. Соберите функцию generate_baseline, которая принимает текст задачи и возвращает предсказание модели с маркером `[ANSWER]` и ответом. Используйте чатовую модель и составьте промпт. Для улучшения можно попробовать прописать промпт с ролью system.

Для воспроизводимости уже выключен сэмплинг. Маленькие модели могут генерировать много текста перед ответом, поэтому ограничьте количество токенов до 64. Чтобы вы отслеживали процесс выполнения, в коде использована функция tqdm для вывода прогресс-бара. 

По умолчанию датасет сохранится в папку с кешем `~/.cache/`. Вы можете поменять путь к этой папке командой:

In [3]:
!export HF_DATASETS_CACHE="~/project/hf_cache"

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from datasets import load_dataset

MODEL_ID = "Qwen/Qwen3-0.6B"
# загрузите токенизатор и модель
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to("cuda") # не забудьте перевести модель на GPU
ds = load_dataset("t-tech/T-math", split="train")

def generate_baseline(question: str) -> str:
    messages = [ # роль system по желанию
        {
        "role": "user",
        "content": (
            "Реши задачу и выведи ответ ТОЛЬКО число после '[ANSWER]'.\n"
            f"Задача: {question}\n"
        ),
    }]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True, # токенизируем
        add_generation_prompt=True, # ставим спецтокен начала ответа
        enable_thinking=False, # разберём этот параметр позже
        return_tensors='pt'
    )
    input_ids = inputs["input_ids"]
    with torch.no_grad():
        # для воспроизводимости выключим сэмплинг
        out = model.generate(input_ids=input_ids.to(model.device),
                             max_new_tokens=64,
                             do_sample=False)
    # очистим ответ от спецтокенов
    # достаём текст из списка индексом 0
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0] 

preds_raw = [generate_baseline(r["question"]) for r in tqdm(ds)]
refs = [r["verifiable_answer"] for r in ds]
print("Пример предсказания:\n", preds_raw[0])

100%|██████████| 331/331 [02:20<00:00,  2.35it/s]

Пример предсказания:
 user
Реши задачу и выведи ответ ТОЛЬКО число после '[ANSWER]'.
Задача: На острове Невезения с населением 96 человек провели пять реформ. Каждой реформой недовольна ровно половина граждан. Гражданин выходит на митинг, если недоволен более чем половиной реформ. Какое максимальное число людей может быть на митинге?

assistant
<think>

</think>

[ANSWER] 128


In [12]:
print(compute_metrics(preds_raw, refs)) 

{'format_rate': 0.7552870090634441, 'accuracy': 0.04229607250755287, 'count': 331}


## 4. Улучшаем zero-shot — few-shot
Если показать человеку примеры решения похожих задач, он обычно справляется лучше. Попробуем применить этот подход к модели через метод `few-shot`. 

В `few‑shot` мы добавляем в промпт несколько «демонстраций»: пары «вопрос → ответ». Эти примеры должны быть похожи по стилю и сложности на целевые. Нельзя допускать, чтобы в примерах была целевая задача.

Для тестирования подхода будем выбирать примеры случайно, но зафиксируем `random seed` для воспроизводимости. Все примеры склеим и отправим одним сообщением.

Помните: чем больше примеров, тем дольше генерация. Начнём с трёх.

In [2]:
import torch
import random
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from datasets import load_dataset

SEED = 42
random.seed(SEED)

MODEL_ID = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto"
)

# Загружаем датасет
ds = load_dataset("t-tech/T-math", split="train")

def generate_few_shot(question: str, idx: int, n_examples: int = 3) -> str:
    """
    question: текущий вопрос
    idx: индекс вопроса в датасете
    n_examples: сколько few-shot примеров использовать
    """
    # Выбираем случайные примеры, исключая текущий
    candidates = list(range(len(ds)))
    candidates.remove(idx)
    sample_idxs = random.sample(candidates, n_examples)

    # Формируем few-shot часть
    few_shot_prompt = "Реши задачу и выведи ответ ТОЛЬКО число после '[ANSWER]'.\n\n"
    for ex_idx in sample_idxs:
        ex = ds[ex_idx]
        few_shot_prompt += (
            f"Задача: {ex['question']}\n"
            f"[ANSWER] {ex['verifiable_answer']}\n\n"
        )

    # Добавляем текущую задачу
    few_shot_prompt += f"Задача: {question}\n"

    # Запаковываем в формат чата
    messages = [{
        "role": "user",
        "content": few_shot_prompt
    }]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors='pt'
    )
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs["input_ids"].to(model.device),
            max_new_tokens=64,
            do_sample=False
        )
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]

preds_raw = [
    generate_few_shot(r["question"], i, n_examples=3)
    for i, r in enumerate(tqdm(ds))
]
refs = [r["verifiable_answer"] for r in ds]

print("Пример предсказания:\n", preds_raw[0])

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1165.08it/s]
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
100%|██████████| 331/331 [02:58<00:00,  1.86it/s]

Пример предсказания:
 user
Реши задачу и выведи ответ ТОЛЬКО число после '[ANSWER]'.

Задача: В соревновании по настольному теннису участвовало ровно 50 ребят, среди которых половина рыцари, всегда говорящие правду, и половина - лжецы, которые всегда лгут. По правилам турнира проигравший выбывал. В результате после нескольких игр ровно половина ребят выбыла. После этого событий каждый из оставшихся участников заявил, что выиграл ровно у одного рыцаря. Какое наибольшее количество рыцарей могло остаться участниками турнира?
[ANSWER] 12

Задача: Пусть $ x $ и $ y $ — пятизначные числа, в десятичной записи которых использованы все десять цифр ровно по одному разу. Найдите наибольшее возможное значение $ x $, если $ \operatorname{tg} x^\circ - \operatorname{tg} y^\circ = 1 + \operatorname{tg} x^\circ \operatorname{tg} y^\circ $ ($ x^\circ $ обозначает угол в $ x $ градусов).
[ANSWER] 98721

Задача: Вася на остановке увидел 1 автобус и 2 трамвая. Шпион, пришедший позже, за время наблюдения у

In [5]:
print(compute_metrics(preds_raw, refs)) 

{'format_rate': 1.0, 'accuracy': 0.0, 'count': 3}


Вы можете протестировать генерацию с бóльшим количеством примеров. 

Судя по метрике format_rate, модель теперь всегда понимает правила генерации ответа. Точность (accuracy) немного упала. Возможно, формат непривычен для модели, так как мы подавали все примеры в одном сообщении.

Идея few-shot в том, что модель должна повторять примеры. Но модель видит текст вместе со специальными токенами, поэтому в каждом примере их нужно расставлять так же, как в сгенерированном ответе.

## Задание 3
Измените код так, чтобы каждая задача была сообщением с ролью `user`, а ответ — с ролью `assistant`. Для этого в при формировании `few shot` примеров в `messages` нужно добавить сначала контент '`f"Задача: {ex\['question'\]}"` с ролью `“user“`, а потом контент `f"\[ANSWER\] {ex\['verifiable_answer'\]}"`  с ролью `"assistant"`. 

In [11]:
import torch
import random
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from datasets import load_dataset

SEED = 42
random.seed(SEED)

MODEL_ID = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto"
)

# Загружаем датасет
ds = load_dataset("t-tech/T-math", split="train")

def generate_few_shot(question: str, idx: int, n_examples: int = 3) -> str:
    """
    question: текущий вопрос
    idx: индекс вопроса в датасете
    n_examples: сколько few-shot примеров использовать
    """
    # Выбираем случайные примеры, исключая текущий
    candidates = list(range(len(ds)))
    candidates.remove(idx)
    sample_idxs = random.sample(candidates, n_examples)

    # Формируем диалог few-shot
    # Отнесём инструкцию в роль system, чтобы отделить от примеров
    messages = [
        {
            "role": "system",
            "content": "Реши задачу и выведи ответ ТОЛЬКО числом после '[ANSWER]'."
        }
    ]
    
    # цикл с добавлением примеров как отдельных сообщений
    for ex_idx in sample_idxs:
        ex = ds[ex_idx]
        messages.append({
            "role": "user",
            "content": f"Задача: {ex['question']}",
        })
        messages.append({
            "role": "assistant",
            "content": f"[ANSWER] {ex['verifiable_answer']}"
        })

    # Добавляем текущую задачу как отдельное сообщение
    messages.append({
        "role": "user",
        "content": f"Задача: {question}",
    })
    # Токенизация и генерация
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors='pt'
    )
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs["input_ids"].to(model.device),
            max_new_tokens=64,
            do_sample=False
        )
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]

preds_raw = [
    generate_few_shot(r["question"], i, n_examples=3)
    for i, r in enumerate(tqdm(ds))
]
refs = [r["full_answer"] for r in ds]

print("Пример предсказания:\n", preds_raw[0])

100%|██████████| 331/331 [03:04<00:00,  1.80it/s]

Пример предсказания:
 system
Реши задачу и выведи ответ ТОЛЬКО числом после '[ANSWER]'.
user
Задача: В соревновании по настольному теннису участвовало ровно 50 ребят, среди которых половина рыцари, всегда говорящие правду, и половина - лжецы, которые всегда лгут. По правилам турнира проигравший выбывал. В результате после нескольких игр ровно половина ребят выбыла. После этого событий каждый из оставшихся участников заявил, что выиграл ровно у одного рыцаря. Какое наибольшее количество рыцарей могло остаться участниками турнира?
assistant
[ANSWER] 12
user
Задача: Пусть $ x $ и $ y $ — пятизначные числа, в десятичной записи которых использованы все десять цифр ровно по одному разу. Найдите наибольшее возможное значение $ x $, если $ \operatorname{tg} x^\circ - \operatorname{tg} y^\circ = 1 + \operatorname{tg} x^\circ \operatorname{tg} y^\circ $ ($ x^\circ $ обозначает угол в $ x $ градусов).
assistant
[ANSWER] 98721
user
Задача: Вася на остановке увидел 1 автобус и 2 трамвая. Шпион, при

In [12]:
print(compute_metrics(preds_raw, refs)) 

{'format_rate': 1.0, 'accuracy': 0.006042296072507553, 'count': 331}


Метрика формата осталась высокой, но точность не изменилась. На практике few-shot в формате отдельных сообщений (multiturn) иногда немного увеличивает метрики, но это не правило, а эмпирическое  наблюдение. 

Получается, просто показать примеры ответов достаточно, чтобы модель увидела нужный формат, но недостаточно, чтобы повысить точность. Приведём аналогию: посмотреть на правильные ответы к похожим задачам не поможет решить её правильно. Помогает смотреть на решения и строить их аналогично. Давайте добавим часть с решением в нашу систему.

## 5. Chain-of-Thoughts (CoT)

В наших экспериментах мы надеялись, что модель «внутри» сравнит решение с задачей и выдаст правильный ответ. Но мы не можем это проверить. Поэтому, как преподаватели просят студентов аргументировать ответ, так и мы попросим модель написать решение.

Логика здесь такая: во время обучения модели с ростом времени тренировки растёт её качество. С увеличением времени на генерацию (на написание решения) качество тоже должно вырасти.

Доработаем наш бейзлайн: попросим модель сначала сгенерировать метку `[SOLUTION]` и написать решение по шагам — метод `Chain-of-Thoughts` (цепочка мыслей). В конце всё так же попросим сгенерировать ответ после метки `[ANSWER]`, только теперь основываясь на решении. Авторы этого метода сразу используют few-shot с примерами решений, чтобы модель понимала формат, который от неё ждут.

Если раньше мы генерировали только ответ, то теперь ещё и решение, а это значительно больше токенов. Поэтому сначала ускорим нашу генерацию — через батчирование запросов.

Ключевые изменения
- добавляем пэддинг-токен, как делали в предыдущих уроках;
- назначаем пэддинг слева, потому что генерация идёт направо;
- вычисляем attention-маску для пэддингов через прямой вызов токенизатора `tokenizer()`;
- формируем батч текстов вместо одного текста.

In [13]:
import torch
import random
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from datasets import load_dataset

SEED = 42
random.seed(SEED)

MODEL_ID = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto"
).eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ds = load_dataset("t-tech/T-math", split="train")

def build_messages(question: str, idx: int, n_examples: int = 3):
    """
    Собирает чат-сообщения для few-shot как в исходной версии.
    """
    candidates = list(range(len(ds)))
    candidates.remove(idx)
    sample_idxs = random.sample(candidates, n_examples)

    messages = [
        {
            "role": "system",
            "content": "Реши задачу и выведи ответ ТОЛЬКО числом после '[ANSWER]'."
        }
    ]
    for ex_idx in sample_idxs:
        ex = ds[ex_idx]
        messages.append({
            "role": "user",
            "content": f"Задача: {ex['question']}"
        })
        messages.append({
            "role": "assistant",
            "content": f"[ANSWER] {ex['verifiable_answer']}"
        })
    messages.append({
        "role": "user",
        "content": f"Задача: {question}"
    })
    return messages

def generate_batch(batch_indices, n_examples: int = 3):
    """
    Батчевая генерация по списку индексов в датасете.
    """
    batch_messages = [
        build_messages(ds[i]["question"], i, n_examples) for i in batch_indices
    ]

    chat_inputs = tokenizer.apply_chat_template(
        batch_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(chat_inputs, 
                       padding=True,
                       return_tensors='pt')

    with torch.no_grad():
        out = model.generate(
            **inputs.to(model.device),
            max_new_tokens=64,
            do_sample=False
        )
    return tokenizer.batch_decode(out, skip_special_tokens=True)

BATCH_SIZE = 4
preds_raw = []

for start in tqdm(range(0, len(ds), BATCH_SIZE)):
    batch_idxs = list(range(start, min(start + BATCH_SIZE, len(ds))))
    preds_raw.extend(generate_batch(batch_idxs, n_examples=3))

refs = [r["verifiable_answer"] for r in ds]

print(compute_metrics(preds_raw, refs))

100%|██████████| 83/83 [02:32<00:00,  1.84s/it]

{'format_rate': 1.0, 'accuracy': 0.006042296072507553, 'count': 331}


## Задание 4
В код добавился промпт о том, что нужно написать решение по шагам. Добавьте в примеры `few-shot` решения после метки `[SOLUTION]`. В датасете они есть в колонке `solutions`. Чтобы уменьшить вероятность зацикливаний, включите семплинг.

Вам нужно:
- Добавить `f"\[SOLUTION\] {ex\['solutions'\]\[0\]} \[ANSWER\] {ex\['verifiable_answer'\]}"` c ролью `"assistant"` при формировании `few shot` примеров.
- Включить семплинг при вызове `model.generate` c помощью параметра `do_sample`.

In [4]:
import torch
import random
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from datasets import load_dataset

SEED = 42
random.seed(SEED)

MODEL_ID = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto"
).eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ds = load_dataset("t-tech/T-math", split="train")

def build_messages(question: str, idx: int, n_examples: int = 3):
    """
    Собирает чат-сообщения для few-shot как в исходной версии.
    """
    candidates = [i for i in range(len(ds)) if len(ds[i]['solutions']) > 0 and i != idx]
    sample_idxs = random.sample(candidates, n_examples)

    messages = [
        {
            "role": "system",
            "content": "Напиши '[SOLUTION]' и после него решение задачи последовательно по шагам. "
                       "Напиши '[ANSWER]' и выведи ответ ТОЛЬКО число.\n"
        }
    ]
    for ex_idx in sample_idxs:
        ex = ds[ex_idx]
        messages.append({
            "role": "user",
            "content": f"Задача: {ex['question']}"
        })
        messages.append({
            "role": "assistant",
            "content": \
                f"[SOLUTION] {ex['solutions'][0]}"\
                "\n"\
                f"[ANSWER] {ex['verifiable_answer']}",
        })
    messages.append({
        "role": "user",
        "content": f"Задача: {question}"
    })
    return messages

def generate_batch(batch_indices, n_examples: int = 3):
    """
    Батчевая генерация по списку индексов в датасете.
    Возвращает список декодированных ответов (как и раньше, с промптом).
    """
    batch_messages = [
        build_messages(ds[i]["question"], i, n_examples) for i in batch_indices
    ]

    chat_inputs = tokenizer.apply_chat_template(
        batch_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(chat_inputs, 
                       padding=True,
                       return_tensors='pt')

    with torch.no_grad():
        out = model.generate(
            **inputs.to(model.device),
            max_new_tokens=64,
            do_sample=True,
        )
    return tokenizer.batch_decode(out, skip_special_tokens=True)

BATCH_SIZE = 4
preds_raw = []

for start in tqdm(range(0, len(ds), BATCH_SIZE)):
    batch_idxs = list(range(start, min(start + BATCH_SIZE, len(ds))))
    preds_raw.extend(generate_batch(batch_idxs, n_examples=3))

refs = [r["verifiable_answer"] for r in ds]

print("Пример предсказания:\n", preds_raw[0])

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

  0%|          | 0/83 [00:00<?, ?it/s]

Пример предсказания:
 system
Напиши '[SOLUTION]' и после него решение задачи последовательно по шагам. Напиши '[ANSWER]' и выведи ответ ТОЛЬКО число.

user
Задача: Дан треугольник ABC. Пусть точка I — центр его вписанной окружности, а точки P и Q — середины сторон AB и AC соответственно. Оказалось, что ∠PIQ + ∠BIC = 180°. Найдите длину отрезка BC, если AB = 20 и AC = 14.
assistant
[SOLUTION] Из условия следует, что ∠BIP + ∠CIQ = 180°. Кроме того, отметим, что PQ || BC как средняя линия треугольника.

Проведём к вписанной окружности треугольника \(ABC\) касательную, параллельную отрезку \(BC\). Обозначим через \(P'\) и \(Q'\) точки пересечения этой касательной со сторонами \(AB\) и \(AC\) соответственно. Поскольку в трапецию \(P'Q'CB\) вписана окружность с центром \(I\), то точка \(I\) является точкой пересечения биссектрис всех четырёх углов этой трапеции. Так как \(P'Q' \parallel BC\), то \(\angle P'Q'C + \angle BCQ' = 180^\circ\). Тогда \(\angle IQ'C + \angle ICQ' = 90^\circ\), откуд

In [6]:
print(compute_metrics(preds_raw, refs)) 

{'format_rate': 1.0, 'accuracy': 0.015105740181268883, 'count': 331}


## 6. Reasoning-модели

Как мы говорили, модели нужно показывать примеры рассуждений по шагам для качественных результатов. Попробуйте убрать примеры решений — качество упадёт. Но добывать примеры для каждого типа задач сложно, а добавление примеров увеличивает длину промпта и время генерации.

Поэтому появилась идея обучать модели так, чтобы они рассуждали сами. Как обучать такие модели, обсудим в следующих уроках. А пока посмотрим, что из этого вышло.

Есть модели, которые рассуждают перед ответом всегда (например, `deepseek-ai/DeepSeek-R1`). Модели `Qwen3`, которые мы тестируем, позволяют указывать, нужно ли рассуждать. Их рассуждения заключаются в тегах:
- <think> — начало рассуждений;
- </think> — конец рассуждений (после этого должен идти ответ).

Тег в этом контексте — это не специальный токен, а текст, по которому парсится ответ после генерации.

Технически, если в промпте только открытый тег рассуждений (`<think>`), модель сначала сгенерирует рассуждения, потом ответ. А если в промпт сразу добавить и открытый, и закрытый теги без текста внутри, мы показываем модели, что рассуждения уже закончились и можно сразу выдавать ответ. 

На практике модели оказалось сложно контролировать: в режиме «без рассуждений» после закрытого тега они начинают генерировать набор мыслей.

Давайте посмотрим на это в коде. Возьмём наш `zero-shot`-бейзлайн, добавим батчинг (промпт для reasoning-модели будет короче, но генерация остаётся длинной). И включим reasoning через параметр в токенизаторе `enable_thinking=True` — не закрывает тег рассуждений.

In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from datasets import load_dataset
from torch.utils.data import Subset

MODEL_ID = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
)

ds = load_dataset("t-tech/T-math", split="train")

def build_prompt(question: str) -> str:
    messages = [{
        "role": "user",
        "content": (
            "Реши задачу и выведи ответ ТОЛЬКО число после '[ANSWER]'.\n"
            f"Задача: {question}\n"
        ),
    }]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )

def generate_reasoning(prompts, batch_size=1):
    preds = []
    for i in tqdm(range(0, len(prompts), batch_size), desc="Generating"):
        chat_inputs = prompts[i:i+batch_size]
        inputs = tokenizer(
            chat_inputs,
            return_tensors="pt",
            padding=True,
        )

        with torch.no_grad():
            out = model.generate(
                **inputs.to(model.device),
                max_new_tokens=32768,
                do_sample=True
            )
        batch_decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        preds.extend(batch_decoded)
    return preds

micro_ds = Subset(ds, range(10))

prompts = [build_prompt(r["question"]) for r in micro_ds]
preds_raw = generate_reasoning(prompts, batch_size=4)

refs = [r["verifiable_answer"] for r in micro_ds]

print("Пример предсказания:\n", preds_raw[0])

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Generating:   0%|          | 0/3 [00:00<?, ?it/s]

KeyboardInterrupt: 